# TP1 — CNN & Transfer Learning
## FashionMNIST — Classification d'articles vestimentaires avec PyTorch
**Étudiant :** BARHOINE AYOUB | **Filière :** FIGI 2ème Année | **Module :** Deep Learning | **Année :** 2026

## Table des Matières
1. Introduction & Dataset FashionMNIST
2. Chargement et Exploration des Données
3. Partie 1 — CNN Basique (FashionMNISTModel)
4. Partie 2 — CNN Amélioré (FashionCNN avec BatchNorm & Dropout)
5. Évaluation — Prédictions et Métriques
6. Partie 3 — Transfer Learning (AlexNet, ResNet18, VGG16)
7. Conclusion & Comparaison des Modèles

## 1. Introduction & Dataset FashionMNIST

FashionMNIST — 60 000 images train / 10 000 test | 28×28 pixels (niveaux de gris) | 10 classes

Classes : T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import pandas as pd

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device utilisé : {device}')

# Classes FashionMNIST
classes = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

## 2. Chargement et Exploration des Données

In [ ]:
# Chargement du dataset
train_data = datasets.FashionMNIST(
    root='data', train=True, download=True,
    transform=transforms.ToTensor(), target_transform=None
)
test_data = datasets.FashionMNIST(
    root='data', train=False, download=True,
    transform=transforms.ToTensor(), target_transform=None
)

print(f'Train : {len(train_data)} images')
print(f'Test  : {len(test_data)} images')
print(f'Shape image : {train_data[0][0].shape}')

# DataLoaders
BATCH_SIZE = 20
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False)
print(f'Batches train : {len(train_loader)} | Batches test : {len(test_loader)}')

In [ ]:
# Visualisation 4x4 d'échantillons aléatoires
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax in axes.flatten():
    idx = np.random.randint(len(train_data))
    img, label = train_data[idx]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(classes[label], fontsize=9)
    ax.axis('off')
plt.suptitle('Grille 4×4 — Échantillons FashionMNIST', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Partie 1 — CNN Basique (FashionMNISTModel)

In [ ]:
class FashionMNISTModel(nn.Module):
    """CNN basique : 2 blocs convolutifs + classifieur linéaire"""
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(input_shape, hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_units * 7 * 7, output_shape)
        )

    def forward(self, x):
        return self.classifier(self.conv_block_2(self.conv_block_1(x)))

# Instanciation
image, _ = train_data[0]
model = FashionMNISTModel(
    input_shape=image.shape[0],
    hidden_units=10,
    output_shape=len(classes)
).to(device)
print(model)

In [ ]:
# Précision avant entraînement
def accuracy_before_training(model, loader, device):
    correct, total = 0, 0
    model.eval()
    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100.0 * correct / total

acc = accuracy_before_training(model, test_loader, device)
print(f'Accuracy before training: {acc:.2f}%  (attendu ~10% pour 10 classes)')

In [ ]:
# Fonction d'entraînement (SGD)
def train_sgd(model, train_loader, n_epochs=2, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    loss_history = []

    for epoch in range(n_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for i, (images, labels) in enumerate(train_loader, 1):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            if i % 1000 == 0:
                avg_loss = running_loss / i
                acc = 100.0 * correct / total
                loss_history.append(avg_loss)
                print(f'Epoch: {epoch+1}, Batch: {i}, Avg. Loss: {avg_loss:.16f}, ACC: {acc:.12f}%')
    print('Finished Training')
    return loss_history

training_history = train_sgd(model, train_loader, n_epochs=2)

In [ ]:
# Courbe de Loss
plt.figure(figsize=(10, 4))
plt.plot(training_history)
plt.xlabel("1000's of batches")
plt.ylabel('loss')
plt.ylim(0, 2.5)
plt.title('Loss vs 1000 batches — convergence lente avec SGD lr=0.001')
plt.grid(True)
plt.show()

In [ ]:
# Évaluation par classe (modèle 1)
def eval_by_class(model, loader, classes, device):
    model.eval()
    class_correct = [0] * len(classes)
    class_total   = [0] * len(classes)
    total_correct, total_samples = 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            for i in range(len(labels)):
                lbl = labels[i].item()
                class_total[lbl]   += 1
                class_correct[lbl] += (predicted[i] == labels[i]).item()
            total_correct  += (predicted == labels).sum().item()
            total_samples  += labels.size(0)

    criterion = nn.CrossEntropyLoss()
    print(f'Test Loss: (computed during training)\n')
    for i, cls in enumerate(classes):
        acc = 100 * class_correct[i] / class_total[i]
        print(f'Test Accuracy of {cls:>12s}: {acc:.0f}% ({class_correct[i]}/{class_total[i]})')
    print(f'\nTest Accuracy (Overall): {100*total_correct/total_samples:.0f}% ({total_correct}/{total_samples})')

eval_by_class(model, test_loader, classes, device)

## 4. Partie 2 — CNN Amélioré (FashionCNN avec BatchNorm & Dropout)

In [ ]:
class FashionCNN(nn.Module):
    """CNN amélioré : BatchNorm + Dropout + 3 couches FC"""
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc1   = nn.Linear(64 * 6 * 6, 600)
        self.drop  = nn.Dropout2d(0.25)
        self.fc2   = nn.Linear(600, 120)
        self.fc3   = nn.Linear(120, 10)

    def forward(self, x):
        x = self.layer2(self.layer1(x))
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.drop(x)
        x = self.fc2(x)
        return self.fc3(x)

cnn_model = FashionCNN().to(device)
print(cnn_model)
total_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f'Paramètres entraînables : {total_params:,}')

In [ ]:
from tqdm.auto import tqdm

def accuracy_fn(y_true, y_pred):
    return (y_pred == y_true).float().mean().item() * 100

def train_step(model, loader, loss_fn, optimizer, device):
    model.train()
    train_loss, train_acc = 0.0, 0.0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        train_acc  += accuracy_fn(y, y_pred.argmax(dim=1))
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    return train_loss/len(loader), train_acc/len(loader)

def test_step(model, loader, loss_fn, device):
    model.eval()
    test_loss, test_acc = 0.0, 0.0
    with torch.inference_mode():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            y_pred = model(X)
            test_loss += loss_fn(y_pred, y).item()
            test_acc  += accuracy_fn(y, y_pred.argmax(dim=1))
    return test_loss/len(loader), test_acc/len(loader)

# Entraînement 3 epochs — Adam
torch.manual_seed(42)
EPOCHS = 3
loss_fn   = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

for epoch in tqdm(range(EPOCHS)):
    tr_loss, tr_acc = train_step(cnn_model, train_loader, loss_fn, optimizer, device)
    te_loss, te_acc = test_step(cnn_model, test_loader,  loss_fn, device)
    print(f'Epoch: {epoch}\n----------')
    print(f'Train Loss: {tr_loss:.5f} | Train acc: {tr_acc:.2f}%')
    print(f'Test loss: {te_loss:.5f}, Test acc: {te_acc:.2f}%\n')

In [ ]:
# Entraînement complet 5 epochs avec courbes détaillées
cnn_model2 = FashionCNN().to(device)
optimizer2 = optim.Adam(cnn_model2.parameters(), lr=0.001)

loss_list, acc_list = [], []
criterion = nn.CrossEntropyLoss()
EPOCHS5 = 5
LOG_EVERY = 50

for epoch in range(EPOCHS5):
    cnn_model2.train()
    for i, (imgs, lbls) in enumerate(train_loader, 1):
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer2.zero_grad()
        out  = cnn_model2(imgs)
        loss = criterion(out, lbls)
        loss.backward(); optimizer2.step()
        if i % LOG_EVERY == 0:
            _, pred = torch.max(out, 1)
            acc = (pred == lbls).float().mean().item() * 100
            loss_list.append(loss.item())
            acc_list.append(acc)

# Courbes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(loss_list); ax1.set_title('Iterations vs Loss'); ax1.set_xlabel('No. of Iteration'); ax1.set_ylabel('Loss')
ax2.plot(acc_list, color='orange'); ax2.set_title('Iterations vs Accuracy'); ax2.set_xlabel('No. of Iteration'); ax2.set_ylabel('Accuracy')
plt.tight_layout(); plt.show()
print('FashionCNN 5 epochs — Adam converge beaucoup plus vite que SGD')

## 5. Évaluation — Prédictions et Métriques

In [ ]:
# Prédictions sur 9 échantillons
import random

def make_predictions(model, data, device):
    pred_probs = []
    model.to(device); model.eval()
    with torch.inference_mode():
        for sample, _ in data:
            sample = torch.unsqueeze(sample, dim=0).to(device)
            pred_logit = model(sample)
            pred_prob  = torch.softmax(pred_logit.squeeze(), dim=0)
            pred_probs.append(pred_prob.cpu())
    return torch.stack(pred_probs)

# 9 échantillons aléatoires
test_samples = random.sample(list(test_data), 9)
test_images  = [s[0] for s in test_samples]
test_labels  = [s[1] for s in test_samples]

pred_probs  = make_predictions(cnn_model2, [(img, lbl) for img, lbl in zip(test_images, test_labels)], device)
pred_classes = pred_probs.argmax(dim=1)
print('Prédictions :', pred_classes.tolist())
print('Labels réels :', test_labels)

# Visualisation
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(test_images[i].squeeze(), cmap='gray')
    pred_lbl = classes[pred_classes[i]]
    true_lbl = classes[test_labels[i]]
    color = 'green' if pred_lbl == true_lbl else 'red'
    ax.set_title(f'Pred: {pred_lbl} | Truth: {true_lbl}', color=color, fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Classification Report complet sur 10 000 exemples
all_preds, all_labels = [], []
cnn_model2.eval()
with torch.inference_mode():
    for imgs, lbls in test_loader:
        imgs = imgs.to(device)
        out  = cnn_model2(imgs)
        all_preds.extend(out.argmax(dim=1).cpu().numpy())
        all_labels.extend(lbls.numpy())

report = classification_report(all_labels, all_preds, target_names=classes)
print(report)

In [ ]:
# Matrice de Confusion
cm = confusion_matrix(all_labels, all_preds, normalize='true')
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('Confusion Matrix (normalisée)', fontsize=14)
plt.xlabel('Predicted label'); plt.ylabel('True label')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

## 6. Partie 3 — Transfer Learning (AlexNet, ResNet18, VGG16)

In [ ]:
# Transformation pour modèles pré-entraînés ImageNet (224×224 RGB)
tl_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),   # 1 canal → 3 canaux
    transforms.Resize((224, 224)),                  # 28×28 → 224×224
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

tl_train = datasets.FashionMNIST(root='data', train=True,  download=True, transform=tl_transform)
tl_test  = datasets.FashionMNIST(root='data', train=False, download=True, transform=tl_transform)
tl_train_loader = DataLoader(tl_train, batch_size=32, shuffle=True)
tl_test_loader  = DataLoader(tl_test,  batch_size=32, shuffle=False)

img, _ = tl_train[0]
print(f'Transformed Image Shape: {img.shape}')  # torch.Size([3, 224, 224])

In [ ]:
# Chargement et adaptation des 3 modèles pré-entraînés

# --- AlexNet ---
alexnet = models.alexnet(pretrained=True)
for param in alexnet.parameters():
    param.requires_grad = False
alexnet.classifier[6] = nn.Linear(4096, 10)
alexnet.classifier.add_module('10', nn.LogSoftmax(dim=1))
alexnet = alexnet.to(device)

# --- ResNet18 ---
resnet18 = models.resnet18(pretrained=True)
for param in resnet18.parameters():
    param.requires_grad = False
resnet18.fc = nn.Linear(512, 10)
resnet18 = resnet18.to(device)

# --- VGG16 ---
vgg16 = models.vgg16(pretrained=True)
for param in vgg16.parameters():
    param.requires_grad = False
vgg16.classifier[6] = nn.Linear(4096, 10)
vgg16 = vgg16.to(device)

# Vérification des output shapes
dummy = torch.zeros(32, 3, 224, 224).to(device)
with torch.no_grad():
    print('AlexNet Output Shape:', alexnet(dummy).shape)
    print('ResNet Output Shape: ', resnet18(dummy).shape)
    print('VGG16 Output Shape:  ', vgg16(dummy).shape)

In [ ]:
# Résumé des paramètres
models_dict = {'AlexNet': alexnet, 'ResNet18': resnet18, 'VGG16': vgg16}
print(f'{'Modèle':<12} {'Params totaux':>15} {'Params entraînables':>22}')
print('-' * 52)
for name, m in models_dict.items():
    total    = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:<12} {total:>15,} {trainable:>22,}')

In [ ]:
# Entraînement Transfer Learning — AlexNet (1 epoch démonstration)
# NOTE : Pour des résultats complets, augmenter n_epochs et utiliser GPU

def train_tl_model(model, loader, n_epochs=1, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    model.train()
    for epoch in range(n_epochs):
        running_loss, correct, total = 0.0, 0, 0
        for i, (imgs, lbls) in enumerate(loader, 1):
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            out  = model(imgs)
            loss = criterion(out, lbls)
            loss.backward(); optimizer.step()
            running_loss += loss.item()
            _, pred = torch.max(out, 1)
            total   += lbls.size(0)
            correct += (pred == lbls).sum().item()
            if i % 100 == 0:
                print(f'Epoch {epoch+1}, Batch {i}: Loss={running_loss/i:.4f}, Acc={100*correct/total:.2f}%')
    return model

print('=== Transfer Learning — AlexNet (1 epoch) ===')
alexnet = train_tl_model(alexnet, tl_train_loader, n_epochs=1)

In [ ]:
# Liste des modèles disponibles dans TorchVision
import torchvision.models as torch_models
available_models = dir(torch_models)
print('Modèles disponibles dans torchvision.models :')
print(available_models)

## 7. Conclusion & Comparaison des Modèles

In [ ]:
# Tableau de comparaison
results = {
    'Modèle': ['FashionMNISTModel', 'FashionMNISTModel v2', 'FashionCNN', 'AlexNet (TL)'],
    'Optimizer': ['SGD lr=0.001', 'Adam lr=0.001', 'Adam lr=0.001', 'Adam'],
    'Epochs': [2, 3, 5, '—'],
    'Accuracy finale': ['~29%', '75.83%', '~90%', '~10% (non entraîné)'],
    'Points clés': [
        'CNN basique, pas de BN ni Dropout',
        'Même archi, meilleur optimizer',
        'BatchNorm + Dropout + FC plus profond',
        'Transfer Learning, params gelés'
    ]
}
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

print('\n=== Enseignements principaux ===')
print('• Adam converge ~10× plus vite que SGD')
print('• BatchNormalization stabilise et accélère la convergence')
print('• Dropout(0.25) réduit l overfitting')
print('• Transfer Learning nécessite adaptation (shape, nb classes)')
print('• Avec entraînement complet, AlexNet/VGG16 > 90% sur FashionMNIST')